## Step 6: Repeat Identification and Masking
**Input:** Scaffolded assembly from Step 5  
**Output:** Soft-masked genome FASTA in `07-repeats-analysis/rm_out/`; 
repeat landscape HTML; summary table  
**Tools:** RepeatModeler v2 (LTRStruct mode), RepeatMasker v4.2.0, 
Dfam database release 3.9 partition 16 (Fungi)  
**Workflow:** (1) RepeatModeler de novo repeat family identification; 
(2) Combined library (Dfam fungi + de novo); 
(3) RepeatMasker soft-masking  
**Key finding:** 5.11% of genome masked as repetitive DNA; 
4.44% interspersed repeats dominated by LTR retroelements (Gypsy/DIRS1 0.89%) 
and DNA transposons (Tc1-IS630-Pogo 0.48%)  
**Reference:** Materials & Methods Section 5 — Nebli et al. (2025)

# Configuration and Setup

In [ ]:
export NCPUS=128
export NPA=$(echo $NCPUS/4 | bc)
export SN=3RR

In [ ]:
alias tetools="apptainer run docker://dfam/tetools:1.93"

# Data Preparation

In [ ]:
mkdir -p 07-repeats-analysis/assembly
cp 06-scaffolding/ragout_maf_output/sample10-scaffolds_scaffolds.fasta 07-repeats-analysis/assembly

In [ ]:
#!/usr/bin/env python3
import os

# Process the assembly file: convert sequences to uppercase and rename
input_file = "07-repeats-analysis/assembly/sample10_assembly.fasta"
output_file = f"07-repeats-analysis/assembly/{os.getenv('SN')}.fasta"

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        if line.startswith('>'):
            outfile.write(line)
        else:
            outfile.write(line.upper())

# Remove the original file
os.remove(input_file)

# Prepare RepeatMasker Libraries

In [ ]:
mkdir -p 07-repeats-analysis/libraries/famdb
wget -P 07.1-repeats-analysis/libraries/famdb https://www.dfam.org/releases/current/families/FamDB/dfam39_full.0.h5.gz
wget -P 07-repeats-analysis/libraries/famdb https://www.dfam.org/releases/current/families/FamDB/dfam39_full.16.h5.gz
#replace it with
curl -L -C - \
  https://www.dfam.org/releases/current/families/FamDB/dfam39_full.16.h5.gz \
  -o 07.1-repeats-analysis/libraries/famdb/dfam39_full.16.h5.gz

gunzip 07.1-repeats-analysis/libraries/famdb/*.gz

In [ ]:
cp -r A-07-repeats-analysis/libraries/famdb/* \
      07-repeats-analysis/libraries/famdb

In [ ]:
mkdir -p 07-repeats-analysis/RepeatMasker
tetools bash -c "cp -r /opt/RepeatMasker/Libraries 07-repeats-analysis/RepeatMasker/"
ln -f 07-repeats-analysis/libraries/famdb/*.h5 07-repeats-analysis/RepeatMasker/Libraries/famdb/

# De Novo Repeat Identification with RepeatModeler

## Build RepeatModeler Database

In [ ]:
mkdir -p 07-repeats-analysis/repeatmodeler
tetools BuildDatabase -name ${SN} 07-repeats-analysis/assembly/${SN}.fasta

## Run RepeatModeler

In [ ]:
tetools RepeatModeler \
  -database ${SN} \
  -threads $NCPUS \
  -LTRStruct \
  > 07-repeats-analysis/repeatmodeler/${SN}_repeatmodeler.log 2>&1

In [ ]:
# Move RepeatModeler results to our analysis directory
mv ${SN}-families.fa 07-repeats-analysis/repeatmodeler/
mv ${SN}-families.stk 07-repeats-analysis/repeatmodeler/
mv ${SN}-rmod.log 07-repeats-analysis/repeatmodeler/
mv RM_* 07-repeats-analysis/repeatmodeler/
rm ${SN}.*

#  Examine RepeatModeler Results

In [ ]:
# Count the number of repeat families identified
grep -c ">" 07-repeats-analysis/repeatmodeler/${SN}-families.fa
# Look at the first few families
head -20 07-repeats-analysis/repeatmodeler/${SN}-families.fa

# Library Reconfiguration

In [ ]:
BIND="--bind $PWD/07-repeats-analysis/RepeatMasker/Libraries:/opt/RepeatMasker/Libraries"

cp -r 07-repeats-analysis/libraries/famdb/* \
       07-repeats-analysis/RepeatMasker/Libraries/

alias tetools_lib="apptainer run $BIND docker://dfam/tetools:1.93"
tetools_lib bash -c "rm -f /opt/RepeatMasker/Libraries/famdb/rmlib.config && cd /opt/RepeatMasker && ./tetoolsDfamUpdate.pl"

# Combine Dfam Database with Custom Library

In [ ]:
# Extract fungi sequences from Dfam using famdb.py
tetools_lib famdb.py -i /opt/RepeatMasker/Libraries/famdb \
  families --format fasta_name --ancestors --descendants 'Fungi' \
  --include-class-in-name > 07-repeats-analysis/RepeatMasker/Libraries/dfam_fungi.fa

In [ ]:
# Create combined library
cat 07-repeats-analysis/RepeatMasker/Libraries/dfam_fungi.fa \
    07-repeats-analysis/repeatmodeler/${SN}-families.fa \
    > 07-repeats-analysis/RepeatMasker/Libraries/${SN}_combined.fa

# Running RepeatMasker with Combined Libraries

In [ ]:
tetools_lib RepeatMasker \
  -a \
  -pa $NPA \
  -lib /opt/RepeatMasker/Libraries/${SN}_combined.fa \
  -dir 07-repeats-analysis/rm_out \
  -gff \
  07-repeats-analysis/assembly/${SN}.fasta > 07-repeats-analysis/rm_out.log
# alternatively but longerto run, we can use -lib /opt/RepeatMasker/Libraries/${SN}_combined.fa

# Visualizing RepeatMasker Results
## Built-in RepeatMasker Visualization Tools

### Generate Repeat Landscape

In [ ]:
# Generate divergence data
tetools_lib calcDivergenceFromAlign.pl \
  -s 07-repeats-analysis/rm_out/${SN}.fasta.divsum \
  07-repeats-analysis/rm_out/${SN}.fasta.align

# Create repeat landscape plot
tetools_lib createRepeatLandscape.pl \
  -g 50400000 \
  -div 07-repeats-analysis/rm_out/${SN}.fasta.divsum \
  > 07-repeats-analysis/rm_out/${SN}_repeat_landscape.html

In [ ]:
#Summary Statistics
# View the repeat summary table
cat 07-repeats-analysis/rm_out/${SN}.fasta.tbl